# Notebook 7 — Cleaned robustness analysis + fine-tuned/Instruct ensemble (CPU only)

1. **Robustness, cleaned.** Text-only exclusion rules fixed *before* re-analysis (no scores used): paraphrases with meta text, of originals ≤ 3 words, or more than 2× the original's length; typo variants identical to the original. Negation results are also split by form (*not* after an auxiliary vs. the "It is not the case that" prefix).
2. **Ensemble of the fine-tuned model and Instruct zero-shot**, two versions: a fixed 50/50 average (no fitting), and weights fitted on validation answers only. Evaluated on the real test sets (with paired tests vs. fine-tuned alone) and on the robustness sets.

**Before running:** upload `perturb.py` to `MyDrive/asag_revision/`. CPU runtime, ~10 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)

In [ ]:
import numpy as np, pandas as pd, nltk
from sklearn.linear_model import LinearRegression, LogisticRegression
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)
from asag_utils import *
from llm_utils import SCORE_TARGETS, LABEL_ORDER, score_from_logprobs, labels_from_logprobs
from perturb import valid_pairs, negation_form
RES = f'{ROOT}/results'
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
LPM = [f'lp_{s}' for s in SCORE_TARGETS]; LPS = [f'lp_{l}' for l in LABEL_ORDER]
VARIANTS = ['negated', 'keyword_salad', 'swapped', 'typos', 'paraphrase']

def cluster_ci(values, groups, n_boot=2000, seed=SEED):
    values, groups = np.asarray(values, float), np.asarray(groups)
    sets_ = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    rng = np.random.default_rng(seed)
    b = [values[np.concatenate([sets_[i] for i in rng.integers(0, len(sets_), len(sets_))])].mean() for _ in range(n_boot)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

def paired_bootstrap(y, a, b, metric_fn, groups=None, n_boot=2000, seed=SEED):
    y, a, b = np.asarray(y), np.asarray(a), np.asarray(b)
    rng = np.random.default_rng(seed)
    sets_ = None if groups is None else [np.flatnonzero(np.asarray(groups) == g) for g in pd.unique(groups)]
    ma, mb = metric_fn(y, a), metric_fn(y, b); D = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y)) if sets_ is None else np.concatenate([sets_[i] for i in rng.integers(0, len(sets_), len(sets_))])
        A, B = metric_fn(y[idx], a[idx]), metric_fn(y[idx], b[idx]); D.append({k: A[k] - B[k] for k in A})
    D = pd.DataFrame(D)
    return {k: (ma[k] - mb[k], np.percentile(D[k].dropna(), 2.5), np.percentile(D[k].dropna(), 97.5),
                min(1.0, max(2 * min((D[k] <= 0).mean(), (D[k] >= 0).mean()), 1 / n_boot))) for k in ma}

## 1. Mohler ensemble: fitted on each fold's validation answers

In [ ]:
ft = pd.concat([pd.read_csv(f'{RES}/mohler_llama3_ft_f{k}.csv') for k in range(N_FOLDS)])
zs = pd.read_csv(f'{RES}/mohler_llama3_instruct_zs.csv').drop_duplicates('uid').set_index('uid').expected
preds = pd.read_csv(f'{RES}/mohler_preds.csv')
ens_w, fitted = [], {}
for k in range(N_FOLDS):
    v = ft[(ft.fold == k) & (ft.split == 'val')].merge(mo[['uid', 'y']], on='uid')
    v['zs'] = v.uid.map(zs)
    lr = LinearRegression(positive=True).fit(v[['expected', 'zs']], v.y); fitted[k] = lr
    w = lr.coef_ / lr.coef_.sum() if lr.coef_.sum() > 0 else lr.coef_
    ens_w.append({'fold': k, 'w_finetuned': w[0], 'w_instruct': w[1], 'intercept': lr.intercept_})
ens_w = pd.DataFrame(ens_w); ens_w.to_csv(f'{RES}/nb7_mohler_ensemble_weights.csv', index=False)
display(ens_w.round(3))

test = ft[ft.split == 'test'].set_index('uid')
preds['ens_50_50'] = 0.5 * preds.llama3_ft + 0.5 * preds.llama3_instruct_zs
preds['ens_fitted'] = [np.clip(fitted[f].predict(pd.DataFrame({'expected': [e], 'zs': [z]}))[0], 0, 10)
                       for f, e, z in zip(preds.fold, preds.llama3_ft, preds.llama3_instruct_zs)]
preds.to_csv(f'{RES}/mohler_preds.csv', index=False)
rows = [{'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)}
        for s in ['llama3_ft', 'llama3_instruct_zs', 'ens_50_50', 'ens_fitted']]
ens_m = pd.DataFrame(rows); ens_m.to_csv(f'{RES}/mohler_nb7_ensemble_summary.csv', index=False)
display(ens_m[['system', 'pearson', 'pearson_ci', 'mae', 'rmse', 'qwk', 'pearson_fold_std']].round(3))
tests = []
for s in ['ens_50_50', 'ens_fitted']:
    r = paired_bootstrap(preds.y, preds[s], preds.llama3_ft, reg_metrics, groups=preds.group)
    tests.append({'comparison': f'{s} vs llama3_ft', **{f'd_{m}': r[m][0] for m in ['pearson', 'mae', 'qwk']},
                  'd_pearson_ci': f'[{r["pearson"][1]:.3f}, {r["pearson"][2]:.3f}]', 'p_pearson': r['pearson'][3],
                  'd_mae_ci': f'[{r["mae"][1]:.3f}, {r["mae"][2]:.3f}]', 'p_mae': r['mae'][3]})
display(pd.DataFrame(tests).round(4))

## 2. SciEntsBank ensemble: logistic regression on both models' label log-probabilities (fitted on validation), and a 50/50 probability average

In [ ]:
ftS = pd.read_csv(f'{RES}/scientsbank_llama3_ft.csv').drop_duplicates('uid').merge(se[['uid', 'split', 'group', 'label', 'label3', 'label2']], on='uid')
zsS = pd.read_csv(f'{RES}/scientsbank_llama3_instruct_zs.csv').drop_duplicates('uid').set_index('uid')
calS = zsS.join(se.set_index('uid')[['split', 'label']]); calS = calS[calS.split.isin(['train', 'val'])]
qc, _, _ = labels_from_logprobs(calS[LPS].values)
zs_cal = LogisticRegression(max_iter=2000).fit(np.log(np.clip(qc, 1e-12, 1)), calS.label)   # Notebook 2 calibration

def probs_aligned(clf, X):
    p = np.zeros((len(X), len(LABEL_ORDER)))
    for j, c in enumerate(clf.classes_):
        p[:, LABEL_ORDER.index(c)] = clf.predict_proba(X)[:, j]
    return p

def ens_features(lp_ft, lp_zs):
    q1, _, _ = labels_from_logprobs(lp_ft); q2, _, _ = labels_from_logprobs(lp_zs)
    return q1, probs_aligned(zs_cal, np.log(np.clip(q2, 1e-12, 1)))

q_ft, q_zs = ens_features(ftS[LPS].values, zsS.loc[ftS.uid, LPS].values)
X = np.log(np.clip(np.hstack([q_ft, q_zs]), 1e-12, 1))
val = (ftS.split == 'val').values
ens_clf = LogisticRegression(max_iter=4000).fit(X[val], ftS.label.values[val])
_, _, p_fit = labels_from_logprobs(np.log(np.clip(probs_aligned(ens_clf, X), 1e-12, 1)))
_, _, p_avg = labels_from_logprobs(np.log(np.clip(0.5 * q_ft + 0.5 * q_zs, 1e-12, 1)))
sp = pd.read_csv(f'{RES}/scientsbank_preds.csv')
new = pd.DataFrame({'uid': ftS.uid})
for sch in ['5way', '3way', '2way']:
    new[f'ens_fitted_{sch}'] = p_fit[sch]; new[f'ens_50_50_{sch}'] = p_avg[sch]
sp = sp.drop(columns=[c for c in new.columns if c != 'uid' and c in sp]).merge(new, on='uid', how='left')
sp.to_csv(f'{RES}/scientsbank_preds.csv', index=False)

label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
rows, tests = [], []
for sch, col in label_col.items():
    for t in ['test_ua', 'test_uq', 'test_ud']:
        cols = [f'{s}_{sch}' for s in ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted']]
        d = sp[sp.split == t].dropna(subset=cols); g = None if t == 'test_ua' else d.group
        for s in ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted']:
            m = cls_metrics(d[col], d[f'{s}_{sch}'])
            rows.append({'scheme': sch, 'test': t, 'system': s, **m})
        for s in ['ens_50_50', 'ens_fitted']:
            r = paired_bootstrap(d[col], d[f'{s}_{sch}'], d[f'llama3_ft_{sch}'], cls_metrics, groups=g)
            tests.append({'scheme': sch, 'test': t, 'comparison': f'{s} vs llama3_ft',
                          'd_macro_f1': r['macro_f1'][0], 'ci': f'[{r["macro_f1"][1]:.3f}, {r["macro_f1"][2]:.3f}]', 'p': r['macro_f1'][3],
                          'd_weighted_f1': r['weighted_f1'][0], 'p_weighted': r['weighted_f1'][3]})
ens_s = pd.DataFrame(rows); ens_s.to_csv(f'{RES}/scientsbank_nb7_ensemble_summary.csv', index=False)
pd.DataFrame(tests).to_csv(f'{RES}/scientsbank_nb7_ensemble_tests.csv', index=False)
for metric in ['macro_f1', 'weighted_f1']:
    print(metric); display(ens_s.pivot_table(index=['scheme', 'system'], columns='test', values=metric).round(3))
display(pd.DataFrame(tests).round(4))

## 3. Robustness, cleaned, with the ensembles added

In [ ]:
rsM = pd.read_csv(f'{RES}/robustness_set.csv'); rsS = pd.read_csv(f'{RES}/robustness_set_scientsbank.csv')
for name, rs in [('mohler', rsM), ('scientsbank', rsS)]:
    ok = valid_pairs(rs)
    print(name, 'excluded:', rs[~ok].variant.value_counts().to_dict())
    rs['keep'] = ok.values
    orig = rs[rs.variant == 'original'].set_index('orig_uid').student_answer
    rs['neg_form'] = rs.orig_uid.map(orig).astype(str).map(negation_form)

# --- Mohler scores
S = rsM[['rid', 'orig_uid', 'fold', 'group', 'variant', 'keep', 'neg_form']].copy()
for name, f in [('llama3_ft', 'robustness_scores_ft.csv'), ('llama3_instruct_zs', 'robustness_scores_instruct_zs.csv')]:
    z = pd.read_csv(f'{RES}/{f}'); e, _, _ = score_from_logprobs(z[LPM].values)
    S = S.merge(pd.DataFrame({'rid': z.rid, name: e}), on='rid')
S = S.merge(pd.read_csv(f'{RES}/robustness_scores_lexical.csv')[['rid', 'score']].rename(columns={'score': 'lexical_composite'}), on='rid')
S['ens_50_50'] = 0.5 * S.llama3_ft + 0.5 * S.llama3_instruct_zs
S['ens_fitted'] = [np.clip(fitted[f].predict(pd.DataFrame({'expected': [a], 'zs': [b]}))[0], 0, 10)
                   for f, a, b in zip(S.fold, S.llama3_ft, S.llama3_instruct_zs)]
SYS_M = ['llama3_ft', 'llama3_instruct_zs', 'ens_50_50', 'ens_fitted', 'lexical_composite']
orig = S[S.variant == 'original'].set_index('orig_uid')
rows = []
for s in SYS_M:
    for v in VARIANTS + ['negated:auxiliary', 'negated:prefix']:
        base_v, form = (v.split(':') + [None])[:2]
        dv = S[(S.variant == base_v) & S.keep]
        if form: dv = dv[dv.neg_form == form]
        dv = dv.set_index('orig_uid'); delta = dv[s] - orig.loc[dv.index, s]
        lo, hi = cluster_ci(delta.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'n': len(dv), 'mean_delta': delta.mean(), 'delta_ci': f'[{lo:.2f}, {hi:.2f}]',
                     'share_drop_ge3': (delta <= -3).mean(), 'share_abs_delta_le1': (delta.abs() <= 1).mean()})
robM = pd.DataFrame(rows); robM.to_csv(f'{RES}/robustness_summary_clean.csv', index=False)
display(robM.pivot_table(index='variant', columns='system', values='mean_delta').loc[VARIANTS + ['negated:auxiliary', 'negated:prefix'], SYS_M].round(2))
display(robM.round(3))

In [ ]:
# --- SciEntsBank labels
P = rsS[['rid', 'orig_uid', 'group', 'variant', 'keep', 'neg_form']].copy()
zf = pd.read_csv(f'{RES}/robustness_scores_scientsbank_ft.csv').set_index('rid')
zz = pd.read_csv(f'{RES}/robustness_scores_scientsbank_instruct_zs.csv').set_index('rid')
q1, q2 = ens_features(zf.loc[P.rid, LPS].values, zz.loc[P.rid, LPS].values)
for name, q in [('llama3_ft', q1), ('llama3_instruct_zs_cal', q2), ('ens_50_50', 0.5 * q1 + 0.5 * q2),
                ('ens_fitted', probs_aligned(ens_clf, np.log(np.clip(np.hstack([q1, q2]), 1e-12, 1))))]:
    P[name] = np.array(LABEL_ORDER)[q.argmax(1)]
P = P.merge(pd.read_csv(f'{RES}/robustness_scores_scientsbank_lexical.csv')[['rid', 'label_pred']].rename(columns={'label_pred': 'lexical_4'}), on='rid')
SYS_S = ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted', 'lexical_4']
rows = []
for s in SYS_S:
    for v in ['original'] + VARIANTS + ['negated:auxiliary', 'negated:prefix']:
        base_v, form = (v.split(':') + [None])[:2]
        dv = P[(P.variant == base_v) & P.keep]
        if form: dv = dv[dv.neg_form == form]
        still = (dv[s] == 'correct').astype(float)
        lo, hi = cluster_ci(still.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'n': len(dv), 'share_still_correct': still.mean(), 'ci': f'[{lo:.2f}, {hi:.2f}]',
                     'share_contradictory': (dv[s] == 'contradictory').mean(),
                     'share_irrelevant_or_nondomain': dv[s].isin(['irrelevant', 'non_domain']).mean()})
robS = pd.DataFrame(rows); robS.to_csv(f'{RES}/robustness_summary_scientsbank_clean.csv', index=False)
display(robS.pivot_table(index='variant', columns='system', values='share_still_correct')
        .loc[['original'] + VARIANTS + ['negated:auxiliary', 'negated:prefix'], SYS_S].round(3))
display(robS.round(3))

Mohler table: mean change in score (lower is better for negated/keyword_salad/swapped; ≈0 is better for typos/paraphrase). SciEntsBank table: share still labelled `correct` (lower is better for the meaning-destroying variants; close to the `original` row is better for typos/paraphrase).